**Desafío Data Sintética**

**Introducción**
Este documento define el Desafío Técnico para el rol de Coordinador TDM, combinando una presentación formal del objetivo del área con las especificaciones técnicas del ejercicio.

**Objetivo**
La Unidad de Entornos No Productivos – Chapter QA busca garantizar datos adecuados, reproducibles y trazables para pruebas funcionales y E2E, para esto se solicita generar un dataset sintético de clientes bajo un contrato de datos explícito, validarlo contra reglas de negocio y producir artefactos consumibles con trazabilidad (logs, reportes, perfiles).

In [ ]:
# =====================================
# SETUP DEL ENTORNO
# Instalación de dependencias
# =====================================
!pip install pyspark pyyaml faker -q

print("✅ Librerías instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.4 MB/s eta 0:00:00
✅ Librerías instaladas correctamente


In [ ]:
# =====================================
# INICIALIZACIÓN DE SPARK
# =====================================
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("TDM_Challenge") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

4.0.2


Generación de 100–500 registros de clientes sintéticos (Parametrizable)

In [68]:
import random
import math
from datetime import datetime, timedelta

# Parámetros según requerimientos
NUM_REGISTROS = 400
PORC_ERROR = 0.0005  # Representa el 0.05%

# Aseguramos al menos 1 error para trazabilidad de QA
num_errores = max(1, int(NUM_REGISTROS * PORC_ERROR))

# Pre-calculamos los índices que serán "saboteados"
# Usamos un rango de 1 a NUM_REGISTROS para coincidir con tu bucle
indices_con_error = random.sample(range(1, NUM_REGISTROS + 1), num_errores)

print(f"🚨 Registros con error a inyectar: {num_errores}")
print(f"📌 Índices seleccionados para error: {indices_con_error}")

🚨 Registros con error a inyectar: 1
📌 Índices seleccionados para error: [357]


In [14]:
# Crear el YAML corregido y mejorado
yaml_content = """
dataset: clientes_no_productivo
version: 1.1
description: Dataset sintético bancario con soporte multidentificación (EC) y validación cruzada
owner: Unidad de Entornos No Productivos

catalogos:
  tipo_identificacion:
    - CEDULA
    - RUC
    - PASAPORTE

  estado_cliente:
    - ACTIVO
    - INACTIVO

tables:
  clientes:
    primary_key: customer_id

    columns:
      customer_id:
        type: string
        required: true
        unique: true
        pattern: "^Cus[0-9]{3}$"
        description: Identificador único autosecuencial del cliente

      tipo_identificacion:
        type: string
        required: true
        catalog: tipo_identificacion
        description: Define la regla de validación para el campo identificacion

      identificacion:
        type: string
        required: true
        unique: true
        sensitive: true
        description: Número de identificación sintético según tipo documental
        rules:
          - if: "tipo_identificacion == 'CEDULA'"
            pattern: "^[0-9]{10}$"
            validator: "modulo_10_ecuador"

          - if: "tipo_identificacion == 'RUC'"
            pattern: "^[0-9]{13}$"
            validator: "modulo_10_o_13_ruc"

          - if: "tipo_identificacion == 'PASAPORTE'"
            pattern: "^[A-Z0-9]{3,15}$"
            description: Alfanumérico estándar internacional

      nombre:
        type: string
        required: true
        max_length: 50
        sensitive: true
        description: Nombre sintético del cliente

      apellido:
        type: string
        required: true
        max_length: 50
        sensitive: true
        description: Apellido sintético del cliente

      fecha_nacimiento:
        type: date
        required: true
        min_age: 18
        max_age: 90
        format: "%d-%m-%Y"
        sensitive: true
        description: Fecha de nacimiento válida para clientes entre 18 y 90 años

      email:
        type: string
        required: true
        format: email
        sensitive: true
        description: Correo electrónico sintético del cliente

      telefono:
        type: string
        required: true
        pattern: "^09[0-9]{8}$"
        sensitive: true
        description: Teléfono celular sintético del cliente

      direccion:
        type: string
        required: true
        max_length: 100
        sensitive: true
        description: Dirección física del cliente

      fecha_creacion:
        type: timestamp
        required: true
        format: "%d/%m/%Y %H:%M:%S"
        not_future: true
        description: Fecha y hora de creación del registro

      estado_cliente:
        type: string
        required: true
        catalog: estado_cliente
        description: Estado lógico del cliente
"""

with open("contract.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("✅ YAML corregido y creado correctamente")

✅ YAML corregido y creado correctamente


In [15]:
import yaml

#  Cargar el contrato para usarlo como referencia en la generación
with open("contract.yaml", "r", encoding="utf-8") as f:
    contract = yaml.safe_load(f)

print("✅ YAML leído correctamente")
print("Dataset:", contract["dataset"])
print("Versión:", contract["version"])
print("Tablas:", list(contract["tables"].keys()))
print("Columnas de clientes:", list(contract["tables"]["clientes"]["columns"].keys()))
print("Catálogos:", list(contract["catalogos"].keys()))

✅ YAML leído correctamente
Dataset: clientes_no_productivo
Versión: 1.1
Tablas: ['clientes']
Columnas de clientes: ['customer_id', 'tipo_identificacion', 'identificacion', 'nombre', 'apellido', 'fecha_nacimiento', 'email', 'telefono', 'direccion', 'fecha_creacion', 'estado_cliente']
Catálogos: ['tipo_identificacion', 'estado_cliente']


In [19]:
#Validación del contrato de datos YAML
print("Primary key:", contract["tables"]["clientes"]["primary_key"])
print("\nRegla customer_id:")
print(contract["tables"]["clientes"]["columns"]["customer_id"])

print("\nRegla tipo_identificacion:")
print(contract["tables"]["clientes"]["columns"]["tipo_identificacion"])

print("\nRegla identificacion:")
print(contract["tables"]["clientes"]["columns"]["identificacion"])

print("\nCatálogo tipo_identificacion:")
print(contract["catalogos"]["tipo_identificacion"])

print("\nCatálogo estado_cliente:")
print(contract["catalogos"]["estado_cliente"])

Primary key: customer_id

Regla customer_id:
{'type': 'string', 'required': True, 'unique': True, 'pattern': '^Cus[0-9]{3}$', 'description': 'Identificador único autosecuencial del cliente'}

Regla tipo_identificacion:
{'type': 'string', 'required': True, 'catalog': 'tipo_identificacion', 'description': 'Define la regla de validación para el campo identificacion'}

Regla identificacion:
{'type': 'string', 'required': True, 'unique': True, 'sensitive': True, 'description': 'Número de identificación sintético según tipo documental', 'rules': [{'if': "tipo_identificacion == 'CEDULA'", 'pattern': '^[0-9]{10}$', 'validator': 'modulo_10_ecuador'}, {'if': "tipo_identificacion == 'RUC'", 'pattern': '^[0-9]{13}$', 'validator': 'modulo_10_o_13_ruc'}, {'if': "tipo_identificacion == 'PASAPORTE'", 'pattern': '^[A-Z0-9]{3,15}$', 'description': 'Alfanumérico estándar internacional'}]}

Catálogo tipo_identificacion:
['CEDULA', 'RUC', 'PASAPORTE']

Catálogo estado_cliente:
['ACTIVO', 'INACTIVO']


In [21]:
from google.colab import files
files.download("contract.yaml")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

GENERACIÓN DE DATOS SINTÉTICOS

PASO 1: FUNCIONES BASE

In [39]:
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker("es_ES")

# Catálogos ejemplo
tipos_id = ["CEDULA", "RUC", "PASAPORTE"]
estados = ["ACTIVO", "INACTIVO"]

nombres = ["Juan", "María", "Carlos", "Ana", "Luis", "Sofía", "Pedro", "Lucía"]
apellidos = ["Pérez", "Gómez", "Torres", "López", "Martínez", "Ramírez", "Vera", "Mora"]

PASO 2: Generar cédula ecuatoriana válida

In [40]:
def calcular_digito_verificador_cedula(base9: str) -> int:
    """
    Calcula el dígito verificador para una cédula ecuatoriana
    a partir de los primeros 9 dígitos.
    """
    coeficientes = [2, 1, 2, 1, 2, 1, 2, 1, 2]
    suma = 0

    for i, digito in enumerate(base9):
        valor = int(digito) * coeficientes[i]
        if valor >= 10:
            valor -= 9
        suma += valor

    digito_verificador = (10 - (suma % 10)) % 10
    return digito_verificador

In [41]:
def generar_cedula_ec(provincia: str = "17") -> str:
    """
    Genera una cédula ecuatoriana sintética válida.
    Por defecto usa provincia 17 (Pichincha / Quito).
    """
    if provincia not in [f"{i:02d}" for i in range(1, 25)]:
        raise ValueError("La provincia debe estar entre 01 y 24.")

    tercer_digito = str(random.randint(0, 5))
    siguientes_seis = "".join(str(random.randint(0, 9)) for _ in range(6))

    base9 = provincia + tercer_digito + siguientes_seis
    verificador = calcular_digito_verificador_cedula(base9)

    return base9 + str(verificador)

PASO 3. Validar cédula ecuatoriana

In [42]:
def validar_cedula_ec(cedula: str) -> bool:
    """
    Valida una cédula ecuatoriana de 10 dígitos.
    """
    if not cedula.isdigit() or len(cedula) != 10:
        return False

    provincia = int(cedula[:2])
    tercer_digito = int(cedula[2])

    if provincia < 1 or provincia > 24:
        return False

    if tercer_digito < 0 or tercer_digito > 5:
        return False

    base9 = cedula[:9]
    verificador_real = int(cedula[9])
    verificador_calculado = calcular_digito_verificador_cedula(base9)

    return verificador_real == verificador_calculado

PASO 4. Generar RUC natural válido para tu caso

In [44]:
def generar_ruc_natural_ec(provincia: str = "17") -> str:
    """
    Genera un RUC sintético de persona natural:
    cédula válida + 001
    """
    return generar_cedula_ec(provincia) + "001"

PASO 5. Generar pasaporte sintético

In [45]:
def generar_pasaporte() -> str:
    """
    Genera un pasaporte sintético alfanumérico.
    """
    return "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", k=8))

PASO 6. Generar identificación según tipo

In [61]:
def generar_identificacion(tipo: str) -> str:
    if tipo == "CEDULA":
        return generar_cedula_ec("17")
    elif tipo == "RUC":
        return generar_ruc_natural_ec("17")
    elif tipo == "PASAPORTE":
        return generar_pasaporte()
    else:
        raise ValueError(f"Tipo de identificación no soportado: {tipo}")

PASO 7: FECHA DE NACIMIENTO (18–90) Se aplica Regla 1: Edad ≥ 18

In [71]:
def generar_fecha_nacimiento(tiene_error=False):
    """
    Regla 1: Edad >= 18 años.
    Si tiene_error es True, genera un menor de edad (entre 1 y 17 años).
    """
    if tiene_error:
        # Sabotaje: Generamos un menor de edad para QA
        edad = random.randint(1, 17)
    else:
        # Happy Path: Cumple la Regla 1
        edad = random.randint(18, 90)

    # Calculamos la fecha restando los años a la fecha actual
    # Usamos 365.25 para mayor precisión con bisiestos
    fecha = datetime.now() - timedelta(days=int(edad * 365.25))

    # Ajuste aleatorio de días para que no todos cumplan años el mismo día
    fecha -= timedelta(days=random.randint(0, 364))

    return fecha.strftime("%d-%m-%Y")

PASO 8: GENERAR EMAIL
Regla 3 (Email Coherente y Controlado)

In [77]:
import unicodedata

def limpiar_texto(texto):
    """Elimina tildes y caracteres especiales para el email."""
    texto = texto.lower().replace(" ", "")
    # Normalización para quitar tildes (Ej: Lucía -> lucia)
    return "".join(c for c in unicodedata.normalize('NFD', texto)
                  if unicodedata.category(c) != 'Mn')

def generar_email_coherente(nombre, apellido, i, es_registro_error=False):
    """
    Regla 3: Email debe tener formato válido y ser coherente con el nombre.
    """
    # 1. Limpieza profesional (quita espacios y tildes)
    n_limpio = limpiar_texto(nombre)
    a_limpio = limpiar_texto(apellido)
    dominio = "testmail.com"

    if es_registro_error:
        # INYECCIÓN DE ERROR: Formatos que QA debe detectar
        opciones_error = [
            f"{n_limpio}.{a_limpio}{i}_sin_arriba.com", # Falta @
            f"{n_limpio}.{a_limpio}{i}@",              # Falta dominio
            f"error_formato_{i}"                       # Texto basura
        ]
        return random.choice(opciones_error)

    # HAPPY PATH: juan.perez1@testmail.com
    return f"{n_limpio}.{a_limpio}{i}@{dominio}"

PASO 9: GENERAR DIRECCION

In [67]:
calles_ec = [
    "Av. Amazonas", "Av. 6 de Diciembre", "Av. Colón",
    "Av. Naciones Unidas", "Av. República", "Av. Shyris",
    "Calle Bolívar", "Calle Sucre", "Av. 9 de Octubre"
]

referencias_ec = [
    "y Naciones Unidas", "y Colón", "y Patria",
    "y 10 de Agosto", "y Orellana", "y Mariana de Jesús",
    "y Loja", "y Sucre"
]

ciudades_ec = ["Quito", "Guayaquil", "Cuenca", "Manta", "Ambato"]

def generar_direccion() -> str:
    calle = random.choice(calles_ec)
    referencia = random.choice(referencias_ec)
    numero = random.randint(100, 9999)
    ciudad = random.choice(ciudades_ec)

    direccion = f"{calle} N{numero} {referencia}, {ciudad}"
    return direccion[:100]



PASO 10: Regla 2: Coherencia de Clientes Inactivos (≥ 6 meses) Para cumplir con el 0.05% de error, se incluye una lógica de "sabotaje" para que, cuando toque inyectar un error, un cliente INACTIVO aparezca con una fecha de creación muy reciente (rompiendo la regla).

In [74]:
def generar_fechas_y_estado(tiene_error=False):
    """
    Regla 2: Si el cliente está INACTIVO, fecha_creacion debe ser >= 6 meses.
    Implementa también el sabotaje para la inyección de errores.
    """
    estado = random.choice(["ACTIVO", "INACTIVO"])
    ahora = datetime.now()

    if tiene_error:
        # SABOTAJE: Rompemos la Regla 2
        # Creamos un INACTIVO con fecha de creación de hace apenas 1 día
        estado = "INACTIVO"
        fecha_creacion = ahora - timedelta(days=1)
    else:
        # HAPPY PATH: Cumplimos la regla
        if estado == "INACTIVO":
            # Debe tener al menos 180 días (6 meses) de antigüedad
            dias_antiguedad = random.randint(180, 730) # Entre 6 meses y 2 años
            fecha_creacion = ahora - timedelta(days=dias_antiguedad)
        else:
            # Si es ACTIVO, puede ser una creación reciente (de 0 a 179 días)
            fecha_creacion = ahora - timedelta(days=random.randint(0, 179))

    return estado, fecha_creacion.strftime("%d/%m/%Y %H:%M:%S")

PASO 11 Regla 4: customer_id Único (y su Sabotaje)

In [80]:
def generar_customer_id(i, tiene_error=False):
    """
    Regla 4: customer_id debe ser único.
    Si hay error, forzamos un duplicado.
    """
    if tiene_error:
        # SABOTAJE: Forzamos que este registro use el ID del primer cliente
        # Esto creará un error de duplicidad que Spark debe detectar
        return "Cus001"

    #  Genera Cus001, Cus002, etc.
    return f"Cus{str(i).zfill(3)}"

PASO 12. INYECCION DE FALLAS Script de Inyección de Fallas (Lógica Centralizada)


In [86]:
def inyectar_falla_especifica(cliente, i):
    """
    Punto 5: Inyecta fallas determinísticas según los 4 tipos requeridos.
    """
    tipos_error = ["schema", "domain", "dup", "business"]
    # Elegimos un tipo de error basado en el índice para que sea determinístico
    falla = tipos_error[i % len(tipos_error)]

    if falla == "schema":
        # Error de Schema: Violación de tipo de dato o nulo (Regla 5)
        cliente["identificacion"] = None
        cliente["nombre"] = 12345  # Error de tipo: número en lugar de string

    elif falla == "domain":
        # Error de Dominio: Valor fuera del catálogo permitido
        cliente["tipo_identificacion"] = "CEDULA_FALSA"
        cliente["estado_cliente"] = "DESCONOCIDO"

    elif falla == "dup":
        # Error de Duplicado: Violación de unicidad (Regla 4)
        cliente["customer_id"] = "Cus001"

    elif falla == "business":
        # Error de Negocio: Violación de reglas funcionales (Regla 1 o 2)
        # Ejemplo: Menor de edad (Regla 1) o Inactivo reciente (Regla 2)
        cliente["fecha_nacimiento"] = "01-01-2025"
        cliente["estado_cliente"] = "INACTIVO"
        cliente["fecha_creacion"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

    return cliente, falla

PASO 10: GENERAR CLIENTE

In [87]:
# Lista para auditoría de qué error se inyectó y dónde
log_errores_inyectados = []

def generar_cliente(i: int) -> dict:
    tiene_error = i in indices_con_error

    # 1. Generación normal
    estado, fecha_creacion = generar_fechas_y_estado(False) # Forzamos normalidad inicial
    nombre = random.choice(nombres)
    apellido = random.choice(apellidos)

    cliente = {
        "customer_id": f"Cus{str(i).zfill(3)}",
        "tipo_identificacion": random.choice(tipos_id),
        "identificacion": generar_identificacion("CEDULA"),
        "nombre": nombre,
        "apellido": apellido,
        "fecha_nacimiento": generar_fecha_nacimiento(),
        "email": generar_email_coherente(nombre, apellido, i, False),
        "telefono": "09" + "".join(str(random.randint(0, 9)) for _ in range(8)),
        "direccion": "Calle Falsa 123",
        "fecha_creacion": fecha_creacion,
        "estado_cliente": estado
    }

    # 2. Aplicación del Punto 5: Inyección de Fallas
    if tiene_error:
        cliente, tipo_falla = inyectar_falla_especifica(cliente, i)
        # Guardamos en el log para el reporte final de validación
        log_errores_inyectados.append({"id": cliente["customer_id"], "falla_inyectada": tipo_falla, "indice": i})
        print(f"🚩 Registro {i}: Falla '{tipo_falla}' inyectada con éxito.")

    return cliente

PASO 11: GENERAR DATASET Y PASAR A SPARK

In [94]:
# --- CELDA DE IMPORTACIÓN ---
from pyspark.sql.types import StructType, StructField, StringType


# --- CELDA DE DEFINICIÓN DEL ESQUEMA ---
schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("tipo_identificacion", StringType(), True),
    StructField("identificacion", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("apellido", StringType(), True),
    StructField("fecha_nacimiento", StringType(), True),
    StructField("email", StringType(), True),
    StructField("telefono", StringType(), True),
    StructField("direccion", StringType(), True),
    StructField("fecha_creacion", StringType(), True),
    StructField("estado_cliente", StringType(), True)
])


# 1. GeneraR la lista de diccionarios
data = [generar_cliente(i) for i in range(1, NUM_REGISTROS + 1)]

# 2. CreaR el DataFrame pasando el SCHEMA como segundo argumento
df = spark.createDataFrame(data, schema=schema)

# 3. DespliegUE
df.show(10, False)

🚩 Registro 357: Falla 'domain' inyectada con éxito.
+-----------+-------------------+--------------+------+--------+----------------+---------------------------+----------+---------------+-------------------+--------------+
|customer_id|tipo_identificacion|identificacion|nombre|apellido|fecha_nacimiento|email                      |telefono  |direccion      |fecha_creacion     |estado_cliente|
+-----------+-------------------+--------------+------+--------+----------------+---------------------------+----------+---------------+-------------------+--------------+
|Cus001     |RUC                |1719766840    |Lucía |Gómez   |25-06-1956      |lucia.gomez1@testmail.com  |0981696124|Calle Falsa 123|27/08/2024 07:09:37|INACTIVO      |
|Cus002     |PASAPORTE          |1731232771    |Carlos|Mora    |25-08-1953      |carlos.mora2@testmail.com  |0971991372|Calle Falsa 123|22/07/2024 07:09:37|INACTIVO      |
|Cus003     |CEDULA             |1720048444    |Lucía |López   |23-10-1972      |lucia.l

PASO 12. VALIDACIÓN DETERMINISTICA DE DATOS GENERADOS

In [95]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def validar_y_reportar(df):
    print("🔎 Iniciando Validación Determinística de Datos...")

    total_registros = df.count()
    # Usamos la fecha actual para las reglas de negocio (2026)
    hoy = F.current_date()

   # 1. Clasificación de Errores
    # ---------------------------------------------------------
    # SCHEMA:
    # - Identificacion es nula
    # - O el nombre contiene solo dígitos (usamos RegEx para evitar el error de CAST)
    df_check = df.withColumn("err_schema",
        F.when(
            F.col("identificacion").isNull() |
            F.col("nombre").rlike("^[0-9]+$"),
            1
        ).otherwise(0))

    # DOMAIN: Valores fuera de ACTIVO/INACTIVO o tipos ID inválidos
    df_check = df_check.withColumn("err_domain",
        F.when(~F.col("estado_cliente").isin("ACTIVO", "INACTIVO") |
               ~F.col("tipo_identificacion").isin("CEDULA", "RUC", "PASAPORTE"), 1).otherwise(0))

    # DUP: Unicidad de customer_id
    w = Window.partitionBy("customer_id")
    df_check = df_check.withColumn("err_dup",
        F.when(F.count("customer_id").over(w) > 1, 1).otherwise(0))

    # BUSINESS: Regla 1 (Edad < 18) y Regla 2 (Inactivo < 6 meses)
    df_check = df_check.withColumn("fnac", F.to_date(F.col("fecha_nacimiento"), "dd-MM-yyyy"))
    df_check = df_check.withColumn("fcrea", F.to_date(F.col("fecha_creacion"), "dd/MM/yyyy HH:mm:ss"))

    df_check = df_check.withColumn("err_business", F.when(
        (F.floor(F.months_between(hoy, F.col("fnac"))/12) < 18) |
        ((F.col("estado_cliente") == "INACTIVO") & (F.months_between(hoy, F.col("fcrea")) < 6)),
        1).otherwise(0))

    # 2. Cálculo de Métricas
    # ---------------------------------------------------------
    resumen = df_check.select(
        F.sum("err_schema").alias("S"), F.sum("err_domain").alias("D"),
        F.sum("err_dup").alias("U"), F.sum("err_business").alias("B")
    ).collect()[0]

    err_totales = (resumen['S'] or 0) + (resumen['D'] or 0) + (resumen['U'] or 0) + (resumen['B'] or 0)
    cumplimiento = ((total_registros - err_totales) / total_registros) * 100

    # 3. Reporte Final
    # ---------------------------------------------------------
    print("\n" + "="*45)
    print("      REPORTE DE MÉTRICAS DE CALIDAD TDM      ")
    print("="*45)
    print(f"Total Registros:      {total_registros}")
    print(f"Reglas Evaluadas:     5 (R1, R2, R3, R4, R5)")
    print(f"Errores Totales:      {int(err_totales)}")
    print(f"% Cumplimiento:       {cumplimiento:.2f}%")
    print("-" * 45)
    print(f"Clasificación de Errores:")
    print(f"  - Schema (Nulos/Tipos): {int(resumen['S'] or 0)}")
    print(f"  - Domain (Catálogos):   {int(resumen['D'] or 0)}")
    print(f"  - Dup (Unicidad):       {int(resumen['U'] or 0)}")
    print(f"  - Business (Lógica):    {int(resumen['B'] or 0)}")
    print("-" * 45)
    print("Muestra de Errores Detectados:")
    df_check.filter("err_schema=1 OR err_domain=1 OR err_dup=1 OR err_business=1") \
            .select("customer_id", "estado_cliente", "fecha_nacimiento").show(5)

# EJECUCIÓN FINAL
validar_y_reportar(df)

🔎 Iniciando Validación Determinística de Datos...

      REPORTE DE MÉTRICAS DE CALIDAD TDM      
Total Registros:      400
Reglas Evaluadas:     5 (R1, R2, R3, R4, R5)
Errores Totales:      1
% Cumplimiento:       99.75%
---------------------------------------------
Clasificación de Errores:
  - Schema (Nulos/Tipos): 0
  - Domain (Catálogos):   1
  - Dup (Unicidad):       0
  - Business (Lógica):    0
---------------------------------------------
Muestra de Errores Detectados:
+-----------+--------------+----------------+
|customer_id|estado_cliente|fecha_nacimiento|
+-----------+--------------+----------------+
|     Cus357|   DESCONOCIDO|      31-05-1975|
+-----------+--------------+----------------+



PASO 13. FORMATOS DE SALIDA


In [98]:
SEED = 42
random.seed(SEED)
fake.seed_instance(SEED)

In [99]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
nombre_base = f"clientes_{timestamp}_seed{SEED}"

print(nombre_base)

clientes_20260405_0732_seed42


Exportar CSV con ese nombre

In [101]:
ruta_csv = f"output/{nombre_base}.csv"

df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(ruta_csv)

print("✅ CSV guardado en:", ruta_csv)

✅ CSV guardado en: output/clientes_20260405_0732_seed42.csv


Exportar JSON

In [102]:
ruta_json = f"output/{nombre_base}.json"

df.coalesce(1).write \
    .mode("overwrite") \
    .json(ruta_json)

print("✅ JSON guardado en:", ruta_json)

✅ JSON guardado en: output/clientes_20260405_0732_seed42.json


In [103]:
import os
import shutil
import glob

# Carpeta temporal generada por Spark
ruta_csv_tmp = f"output/{nombre_base}.csv"
ruta_json_tmp = f"output/{nombre_base}.json"

# Buscar archivo real CSV
csv_generado = glob.glob(f"{ruta_csv_tmp}/part-*.csv")[0]
csv_final = f"{nombre_base}.csv"
shutil.copy(csv_generado, csv_final)

# Buscar archivo real JSON
json_generado = glob.glob(f"{ruta_json_tmp}/part-*.json")[0]
json_final = f"{nombre_base}.json"
shutil.copy(json_generado, json_final)

print("✅ Archivo CSV final:", csv_final)
print("✅ Archivo JSON final:", json_final)

✅ Archivo CSV final: clientes_20260405_0732_seed42.csv
✅ Archivo JSON final: clientes_20260405_0732_seed42.json
